# **PHASE 1 — DATA FOUNDATION**
- Let get all trade summary file from local folder
- merge them into single master csv data file

### Scan "C:\Users\Lapmart\Downloads\CSE" folder and crate master data set

# Fetch data from local folder in name 260320_trade-summary

### Create DuckDB Connection

In [34]:
import os
import duckdb

# Database folder
DB_FOLDER = "database"
os.makedirs(DB_FOLDER, exist_ok=True)
DB_PATH = os.path.join(DB_FOLDER, "cse_data.db")
con = duckdb.connect(database=DB_PATH)  # Create/connect DB

LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

## Helpers

In [35]:
# ================ HELPERS =================

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

# Method to extract date from file name in format like 260210_tradesummary.csv.
def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        return None

### Remove exiting database to create new one

In [36]:
try :
    con.close()
    import os
    os.remove(DB_PATH)
    print("closed connection of duckdb & deleted cse_data.db file...")
except Exception as e:
    print("There is no database connectionto close: ", e)


log("Creating DuckDB connection & table 'stocks_table' ...")

con = duckdb.connect(database=DB_PATH)  # Create/connect DB

closed connection of duckdb & deleted cse_data.db file...
Creating DuckDB connection & table 'stocks_table' ...


In [37]:
import pandas as pd
import os
from datetime import datetime
import glob # finds files/folders matching patterns like wildcards in Python


COLUMN_MAP = {
    "Company Name": "company",
    "Symbol": "symbol",
    "Share Volume": "volume",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}



def load_all_csvs_to_single_csv(source_folder, output_file):
    all_dfs = []

    files = glob.glob(os.path.join(source_folder, "*.csv"))
    print(f"Total files found: {len(files)}")

    for file in files:
        filename = os.path.basename(file)
        date = extract_date(filename)

        if date is None:
            print(f"Skipped (invalid date): {filename}")
            continue

        try:
            df = pd.read_csv(file)
            if df.empty:
                continue
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue

        # only light normalization (optional)
        df.columns = [col.strip() for col in df.columns]
        df.rename(columns=COLUMN_MAP, inplace=True)

        df["date"] = date

        all_dfs.append(df)

    if len(all_dfs) == 0:
        print("No valid data found")
        return

    master_df = pd.concat(all_dfs, ignore_index=True)

    master_df.to_csv(output_file, index=False)
    print(f"Raw merged CSV created → {output_file}")

    con = duckdb.connect(database=DB_PATH)

    # temopraly register df in duckdb
    con.register("temp_df", master_df)

    # Create new table & add data into it 
    con.execute("""
        CREATE OR REPLACE TABLE raw_stocks_table AS
        SELECT * FROM temp_df
    """)
    print("DuckDB table created 'raw_stocks_table'")

In [38]:
SOURCE_FOLDER = r"C:\Users\Lapmart\Downloads\CSE" # my local stored folder path

# Exporting into MASTER CSV
CSV_FOLDER = "output_csv"
os.makedirs(CSV_FOLDER, exist_ok=True)
MASTER_CSV_OUTPUT = os.path.join(CSV_FOLDER, "MASTER_CSE_DATA.csv")

load_all_csvs_to_single_csv(SOURCE_FOLDER, MASTER_CSV_OUTPUT)

Total files found: 62
Raw merged CSV created → output_csv\MASTER_CSE_DATA.csv
DuckDB table created 'raw_stocks_table'


In [39]:
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetch_df()

print(tables)

         table_name
0  raw_stocks_table


### Close DuckDB Connection

In [40]:
if 'con' in globals():  # Check if connection exists
    try:
        con.close()        # Close it
        print("DuckDB connection closed.")
    except Exception as e:
        print("Error closing DuckDB:", e)
    finally:
        del con             # Delete the variable from memory

# raw_stocks_table

DuckDB connection closed.
